# Book Reviews NLP: Text Preprocessing, EDA, and Word2Vec

This notebook solves two tasks:

1. **Task 1 — Text Preprocessing and EDA**
   - Clean raw review text.
   - Create `cleaned_review` and `tokens` columns.
   - Analyze rating distribution, review length, and word frequency.
   - Save plots and cleaned data.

2. **Task 2 — Word2Vec Training**
   - Train a Word2Vec model on cleaned reviews.
   - Evaluate word similarities.
   - Visualize word vectors using PCA and t-SNE.
   - Create optional document embeddings.

> Expected input file: `all_data.csv` in the same folder as this notebook.

## 1. Imports, Settings, and Paths

All libraries are imported in this single cell to keep the notebook clean and GitHub-ready.

In [ ]:
# =========================
# Standard libraries
# =========================
import re
import warnings
from pathlib import Path
from collections import Counter

# =========================
# Data handling
# =========================
import numpy as np
import pandas as pd

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False

# =========================
# NLP preprocessing
# =========================
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# =========================
# Word2Vec and vector visualization
# =========================
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# =========================
# Notebook settings
# =========================
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (10, 6)

# =========================
# Project paths
# =========================
DATA_PATH = Path("all_data.csv")
OUTPUT_DIR = Path("outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
TASK2_DIR = OUTPUT_DIR / "word2vec"

OUTPUT_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TASK2_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 2. Download NLTK Resources

These resources are needed for tokenization, stopword removal, and lemmatization.

In [ ]:
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

print("NLTK resources are ready.")

## 3. Load and Inspect the Dataset

We start by loading the dataset and checking its structure, missing values, and key columns.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Place all_data.csv beside this notebook.")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
missing_values = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("Dataset info:")
df.info()

print("\nBasic summary:")
print("Rows:", len(df))
print("Unique books:", df["book_title"].nunique())
print("Unique authors:", df["book_author"].nunique())
print("Unique reviewers:", df["reviewer_name"].nunique())

print("\nRating summary:")
display(df["book_rating"].describe())

print("\nMissing values:")
display(missing_values)

## 4. Text Preprocessing

The preprocessing pipeline removes noise from raw reviews:

- Goodreads `(less)` text
- URLs
- HTML tags
- punctuation, numbers, and non-alphabetic tokens
- stopwords and extra high-frequency noise words
- then applies lemmatization

In [ ]:
# Keep only rows that contain review text
required_columns = ["review", "book_rating"]
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df_nlp = df.dropna(subset=["review"]).copy()

# Raw text length features
_df_review_as_text = df_nlp["review"].astype(str)
df_nlp["raw_char_count"] = _df_review_as_text.apply(len)
df_nlp["raw_word_count"] = _df_review_as_text.apply(lambda text: len(text.split()))

# Stopwords
base_stopwords = set(stopwords.words("english"))
extra_stopwords = {
    "br", "amp", "quot",
    "n't", "'s", "'m", "'re", "'ve", "'ll",
    "would", "could", "also",
    "one", "get", "thing", "even", "make", "really", "know", "much",
    "way", "people", "want", "think", "well",
    "review", "see",
    "u", "ca", "de", "byte", "ymmv"
}
stop_words = base_stopwords.union(extra_stopwords)
lemmatizer = WordNetLemmatizer()


def clean_review(text: str) -> str:
    """Clean one review and return a whitespace-separated cleaned string."""
    text = str(text)
    text = text.replace("(less)", " ")
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.lower()

    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words and len(token) > 1
    ]

    return " ".join(cleaned_tokens)


df_nlp["cleaned_review"] = df_nlp["review"].apply(clean_review)
df_nlp["tokens"] = df_nlp["cleaned_review"].apply(str.split)
df_nlp["cleaned_word_count"] = df_nlp["tokens"].apply(len)

# Remove rows that became empty after preprocessing
df_nlp = df_nlp[df_nlp["cleaned_word_count"] > 0].copy()

print("Original rows:", len(df))
print("Rows after removing missing/empty reviews:", len(df_nlp))
display(df_nlp[["review", "cleaned_review", "raw_word_count", "cleaned_word_count"]].head())

## 5. Save the Preprocessed Dataset

This file is the main deliverable for Task 1 and will also be used for Task 2.

In [ ]:
columns_to_save = [
    col for col in [
        "ID", "book_title", "Book_series", "book_rating", "book_author", "genre", "reviewer_name",
        "review", "cleaned_review", "raw_word_count", "cleaned_word_count"
    ]
    if col in df_nlp.columns
]

preprocessed_path = OUTPUT_DIR / "preprocessed_book_reviews.csv"
df_nlp[columns_to_save].to_csv(preprocessed_path, index=False)

print(f"Saved preprocessed dataset to: {preprocessed_path}")

## 6. EDA: Rating Distribution and Review Length

We analyze whether ratings are balanced and whether review length has a clear relationship with book rating.

In [ ]:
rating_bins = [0, 3.5, 4.0, 4.5, 5.0]
rating_labels = ["<= 3.5", "3.51 - 4.00", "4.01 - 4.50", "4.51 - 5.00"]

df_nlp["rating_group"] = pd.cut(
    df_nlp["book_rating"],
    bins=rating_bins,
    labels=rating_labels,
    include_lowest=True
)

rating_group_counts = df_nlp["rating_group"].value_counts().sort_index()
correlation = df_nlp["book_rating"].corr(df_nlp["raw_word_count"])

print("Rating group counts:")
display(rating_group_counts)

print(f"Correlation between book rating and review word count: {correlation:.4f}")

In [ ]:
pdf_path = PLOTS_DIR / "eda_plots_report.pdf"

with PdfPages(pdf_path) as pdf:
    # Rating histogram
    plt.figure(figsize=(8, 5))
    plt.hist(df_nlp["book_rating"], bins=20, edgecolor="black")
    plt.title("Book Rating Distribution")
    plt.xlabel("Book Rating")
    plt.ylabel("Count")
    plt.grid(axis="y", alpha=0.3)
    plt.savefig(PLOTS_DIR / "rating_distribution_histogram.png", dpi=300, bbox_inches="tight")
    pdf.savefig(bbox_inches="tight")
    plt.show()

    # Rating pie chart
    plt.figure(figsize=(7, 7))
    plt.pie(
        rating_group_counts,
        labels=rating_group_counts.index,
        autopct="%1.1f%%",
        startangle=90
    )
    plt.title("Rating Group Distribution")
    plt.savefig(PLOTS_DIR / "rating_distribution_pie.png", dpi=300, bbox_inches="tight")
    pdf.savefig(bbox_inches="tight")
    plt.show()

    # Review length boxplot
    groups = [df_nlp.loc[df_nlp["rating_group"] == label, "raw_word_count"] for label in rating_labels]
    plt.figure(figsize=(10, 6))
    plt.boxplot(groups, labels=rating_labels, showfliers=False)
    plt.title("Review Word Count by Rating Group")
    plt.xlabel("Rating Group")
    plt.ylabel("Raw Word Count")
    plt.grid(axis="y", alpha=0.3)
    plt.savefig(PLOTS_DIR / "review_length_by_rating_boxplot.png", dpi=300, bbox_inches="tight")
    pdf.savefig(bbox_inches="tight")
    plt.show()

    # Rating vs review length scatter plot
    plt.figure(figsize=(8, 5))
    plt.scatter(df_nlp["book_rating"], df_nlp["raw_word_count"], alpha=0.2)
    plt.title("Book Rating vs Review Word Count")
    plt.xlabel("Book Rating")
    plt.ylabel("Raw Word Count")
    plt.grid(alpha=0.3)
    plt.savefig(PLOTS_DIR / "rating_vs_review_length_scatter.png", dpi=300, bbox_inches="tight")
    pdf.savefig(bbox_inches="tight")
    plt.show()

print(f"EDA plots saved to: {PLOTS_DIR}")
print(f"PDF report saved to: {pdf_path}")

## 7. EDA: Word Frequency Before and After Preprocessing

This comparison shows why preprocessing is important. Before cleaning, the most common words are usually stopwords. After cleaning, the top words should be more meaningful for book reviews.

In [ ]:
# Word frequency before preprocessing
raw_words = []
for review in df_nlp["review"]:
    raw_words.extend(re.findall(r"[a-zA-Z]+", str(review).lower()))
raw_common_words = Counter(raw_words).most_common(20)

# Word frequency after preprocessing
cleaned_words = [token for tokens in df_nlp["tokens"] for token in tokens]
cleaned_common_words = Counter(cleaned_words).most_common(20)

word_freq_before_df = pd.DataFrame(raw_common_words, columns=["word", "frequency"])
word_freq_after_df = pd.DataFrame(cleaned_common_words, columns=["word", "frequency"])

word_freq_before_df.to_csv(OUTPUT_DIR / "word_frequency_before_preprocessing.csv", index=False)
word_freq_after_df.to_csv(OUTPUT_DIR / "word_frequency_after_preprocessing.csv", index=False)

print("Top words before preprocessing:")
display(word_freq_before_df)

print("Top words after preprocessing:")
display(word_freq_after_df)

In [ ]:
# Plot word frequency before preprocessing
plt.figure(figsize=(10, 6))
plt.barh(word_freq_before_df["word"][::-1], word_freq_before_df["frequency"][::-1])
plt.title("Top 20 Words Before Preprocessing")
plt.xlabel("Frequency")
plt.ylabel("Words")
plt.savefig(PLOTS_DIR / "word_frequency_before_preprocessing.png", dpi=300, bbox_inches="tight")
plt.show()

# Plot word frequency after preprocessing
plt.figure(figsize=(10, 6))
plt.barh(word_freq_after_df["word"][::-1], word_freq_after_df["frequency"][::-1])
plt.title("Top 20 Words After Preprocessing")
plt.xlabel("Frequency")
plt.ylabel("Words")
plt.savefig(PLOTS_DIR / "word_frequency_after_preprocessing.png", dpi=300, bbox_inches="tight")
plt.show()

# Optional word cloud
if WORDCLOUD_AVAILABLE:
    all_cleaned_text = " ".join(df_nlp["cleaned_review"])
    wordcloud = WordCloud(width=1000, height=500, background_color="white", max_words=100).generate(all_cleaned_text)

    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud After Preprocessing")
    plt.savefig(PLOTS_DIR / "wordcloud_after_preprocessing.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("wordcloud is not installed. Skipping word cloud.")

## 8. Original vs. Cleaned Review Examples

This table confirms that the preprocessing pipeline is working as expected.

In [ ]:
comparison_df = df_nlp[["review", "cleaned_review"]].head(10).copy()
comparison_df["review"] = comparison_df["review"].str.slice(0, 300)
comparison_df["cleaned_review"] = comparison_df["cleaned_review"].str.slice(0, 300)

comparison_path = OUTPUT_DIR / "original_vs_cleaned_examples.csv"
comparison_df.to_csv(comparison_path, index=False)

display(comparison_df)
print(f"Saved comparison examples to: {comparison_path}")

## 9. Prepare Data for Word2Vec

Word2Vec expects a list of token lists, not a normal text string.

In [ ]:
df_w2v = df_nlp[df_nlp["tokens"].apply(len) >= 3].copy()
sentences = df_w2v["tokens"].tolist()

all_tokens = [token for review_tokens in sentences for token in review_tokens]
token_counts = Counter(all_tokens)

print("Reviews used for Word2Vec:", len(sentences))
print("Total tokens:", len(all_tokens))
print("Unique tokens before min_count filtering:", len(token_counts))
print("Example tokenized review:")
print(sentences[0][:30])

## 10. Train and Save the Word2Vec Model

We use Skip-gram because it usually performs well when learning relationships for meaningful words in text corpora.

In [ ]:
word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,
    negative=10,
    sample=1e-3,
    epochs=20,
    seed=RANDOM_STATE
)

model_path = TASK2_DIR / "text_review_word2vec.model"
word2vec_model.save(str(model_path))

print("Word2Vec training completed.")
print("Vocabulary size after min_count filtering:", len(word2vec_model.wv))
print(f"Model saved to: {model_path}")

## 11. Evaluate Word Similarity

We test the model using important book-review words. Good results should show related words close together, such as `book`, `story`, `novel`, `author`, and `series`.

In [ ]:
query_words = ["book", "story", "character", "love", "series", "author", "novel", "star", "good", "read"]
similarity_rows = []

for query_word in query_words:
    if query_word not in word2vec_model.wv:
        print(f"'{query_word}' is not in the vocabulary.")
        continue

    print("=" * 60)
    print(f"Most similar words to: {query_word}")

    for similar_word, score in word2vec_model.wv.most_similar(query_word, topn=10):
        print(f"{similar_word:20s} {score:.4f}")
        similarity_rows.append({
            "query_word": query_word,
            "similar_word": similar_word,
            "similarity_score": score
        })

similarity_df = pd.DataFrame(similarity_rows)
similarity_path = TASK2_DIR / "word_similarity_results.csv"
similarity_df.to_csv(similarity_path, index=False)

print(f"\nSaved similarity results to: {similarity_path}")
display(similarity_df.head(20))

In [ ]:
word_pairs = [
    ("book", "story"),
    ("book", "novel"),
    ("love", "loved"),
    ("character", "story"),
    ("author", "book"),
    ("series", "book"),
    ("star", "good")
]

pair_results = []
for word_1, word_2 in word_pairs:
    similarity = None
    if word_1 in word2vec_model.wv and word_2 in word2vec_model.wv:
        similarity = float(word2vec_model.wv.similarity(word_1, word_2))

    pair_results.append({
        "word_1": word_1,
        "word_2": word_2,
        "similarity": similarity
    })

pair_similarity_df = pd.DataFrame(pair_results)
pair_similarity_path = TASK2_DIR / "word_pair_similarity.csv"
pair_similarity_df.to_csv(pair_similarity_path, index=False)

display(pair_similarity_df)
print(f"Saved pair similarity results to: {pair_similarity_path}")

## 12. Visualize Word Vectors with PCA and t-SNE

The model creates 100-dimensional vectors. PCA and t-SNE reduce them to 2 dimensions so we can inspect whether related words appear near each other.

In [ ]:
def plot_word_vectors_2d(words, vectors_2d, title, xlabel, ylabel, output_path):
    """Plot 2D word vectors with labels and save the figure."""
    plt.figure(figsize=(14, 10))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.7)

    for index, word in enumerate(words):
        plt.annotate(word, (vectors_2d[index, 0], vectors_2d[index, 1]), fontsize=9)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


# Use the most frequent words that are present in the model vocabulary
top_words = [word for word, _ in token_counts.most_common(100) if word in word2vec_model.wv]
word_vectors = np.array([word2vec_model.wv[word] for word in top_words])

# PCA
pca_vectors = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(word_vectors)
pca_plot_path = TASK2_DIR / "word2vec_pca_top_words.png"
plot_word_vectors_2d(
    top_words,
    pca_vectors,
    "PCA Visualization of Top Word2Vec Word Vectors",
    "PCA Component 1",
    "PCA Component 2",
    pca_plot_path
)

# t-SNE
perplexity = min(30, max(5, (len(top_words) - 1) // 3))
tsne_vectors = TSNE(
    n_components=2,
    perplexity=perplexity,
    random_state=RANDOM_STATE,
    init="pca",
    learning_rate="auto"
).fit_transform(word_vectors)

tsne_plot_path = TASK2_DIR / "word2vec_tsne_top_words.png"
plot_word_vectors_2d(
    top_words,
    tsne_vectors,
    "t-SNE Visualization of Top Word2Vec Word Vectors",
    "t-SNE Component 1",
    "t-SNE Component 2",
    tsne_plot_path
)

print(f"Saved PCA plot to: {pca_plot_path}")
print(f"Saved t-SNE plot to: {tsne_plot_path}")

## 13. Optional: Create Document Embeddings

Each review is represented by averaging the Word2Vec vectors of its words. These embeddings can later be used for clustering, classification, recommendation, or similarity search.

In [ ]:
def get_document_vector(tokens, model):
    """Return the average Word2Vec vector for one tokenized review."""
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)


document_vectors = np.array([
    get_document_vector(tokens, word2vec_model)
    for tokens in df_w2v["tokens"]
])

embedding_columns = [f"embedding_{i}" for i in range(word2vec_model.vector_size)]
document_embeddings_df = pd.DataFrame(document_vectors, columns=embedding_columns)

metadata_columns = [col for col in ["ID", "book_title", "book_rating", "book_author", "genre"] if col in df_w2v.columns]
document_embeddings_df = pd.concat(
    [df_w2v[metadata_columns].reset_index(drop=True), document_embeddings_df],
    axis=1
)

document_embeddings_path = TASK2_DIR / "document_embeddings.csv"
document_embeddings_df.to_csv(document_embeddings_path, index=False)

print("Document embedding matrix shape:", document_vectors.shape)
print(f"Saved document embeddings to: {document_embeddings_path}")
display(document_embeddings_df.head())

## 14. Save Final Training Summary

This markdown summary is useful for GitHub and project review.

In [ ]:
summary_text = f"""
# Word2Vec Training Summary

## Dataset
- Reviews used for training: {len(sentences)}
- Total tokens: {len(all_tokens)}
- Unique tokens before min_count filtering: {len(token_counts)}
- Word2Vec vocabulary size after min_count filtering: {len(word2vec_model.wv)}

## Model Parameters
- vector_size: {word2vec_model.vector_size}
- window: 5
- min_count: 5
- sg: 1  # Skip-gram
- negative: 10
- sample: 1e-3
- epochs: 20

## Main Outputs
- outputs/preprocessed_book_reviews.csv
- outputs/plots/eda_plots_report.pdf
- outputs/word2vec/text_review_word2vec.model
- outputs/word2vec/word_similarity_results.csv
- outputs/word2vec/word_pair_similarity.csv
- outputs/word2vec/word2vec_pca_top_words.png
- outputs/word2vec/word2vec_tsne_top_words.png
- outputs/word2vec/document_embeddings.csv

## Interpretation
The model was trained on cleaned and tokenized book reviews. Similarity queries and vector visualizations show that related words such as book, story, novel, character, love, author, and series appear close together in the learned vector space.
"""

summary_path = TASK2_DIR / "word2vec_training_summary.md"
summary_path.write_text(summary_text, encoding="utf-8")

print(f"Saved training summary to: {summary_path}")

## 15. Generated Files

Run this cell at the end to verify all outputs were created successfully.

In [ ]:
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path)